<a href="https://colab.research.google.com/github/chenghao55/backrooms-rag/blob/RAG01/RAG01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG01 — 建立向量資料庫
本專案[GitHub](https://github.com/chenghao55/backrooms-rag)

本筆記本負責 RAG 的離線索引階段：從 GitHub 下載 Level 37 語意區塊，使用多語言 Sentence Transformers 模型產生正規化向量，再建立 FAISS 餘弦相似度索引並匯出 `rag-data.zip`。

資料流程：`chunks.json` → 資料驗證 → `title + text` embedding → `IndexFlatIP` → 檢索測試 → 可攜式 ZIP。
> 此階段不呼叫 LLM，也不包含 Groq、Gradio 或文字冒險功能。

## 1. 安裝相依套件

在全新的 Google Colab 執行環境中安裝 Sentence Transformers、FAISS 與 HTTP 下載套件。

In [1]:
%pip install -q sentence-transformers faiss-cpu requests

## 2. 設定與 GitHub 資料載入

下方設定取自此專案的 Git remote。若 repository、預設分支或資料路徑改變，只需修改這三個變數。公開 repository 必須能透過 raw GitHub URL 讀取。

chunks 是將raw data依語意、段落拆分後的jason區塊，放在"~/chunks.json"中

Python 解析 JSON 後，JSON object 通常會變成 Python 字典 dict，便能像字典一樣讀取。

`chunks = response.json()`
則是由 requests 將 HTTP 回應中的 JSON 直接解析成 Python 的 list 和 dict。

In [2]:
import json
import shutil
import textwrap
from pathlib import Path

import requests # HTTP/HTTPS

GITHUB_REPO_URL = "https://github.com/chenghao55/backrooms-rag"
GITHUB_BRANCH = "main"
CHUNKS_PATH = "data/chunks.json"

repo_base = GITHUB_REPO_URL.rstrip("/")
if repo_base.endswith(".git"):
    repo_base = repo_base[:-4]
if not repo_base.startswith("https://github.com/"):
    raise ValueError("GITHUB_REPO_URL 必須是 https://github.com/<owner>/<repo> 格式。")

repo_path = repo_base.removeprefix("https://github.com/")
raw_chunks_url = f"https://raw.githubusercontent.com/{repo_path}/{GITHUB_BRANCH}/{CHUNKS_PATH}"
print(f"資料來源：{raw_chunks_url}")

try:
    response = requests.get(raw_chunks_url, timeout=30)
    response.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        "無法下載 chunks.json。請確認 repository 為公開、分支名稱與 CHUNKS_PATH 正確。"
    ) from exc

try:
    chunks = response.json()
except ValueError as exc:
    raise RuntimeError("下載內容不是有效的 JSON。") from exc

print("chunks.json 下載完成。")

資料來源：https://raw.githubusercontent.com/chenghao55/backrooms-rag/main/data/chunks.json
chunks.json 下載完成。


## 3. 驗證資料集 (Vaidation)

在嵌入前檢查根節點、必要欄位、ID 唯一性、文字內容、內容類型與區塊總數。

In [3]:
REQUIRED_FIELDS = {"id", "title", "section", "content_type", "source_url", "text"}
ALLOWED_CONTENT_TYPES = {"canonical", "narrative"}
EXPECTED_CHUNK_COUNT = 10

if not isinstance(chunks, list):
    raise TypeError("資料集根節點必須是 JSON list。")
if len(chunks) != EXPECTED_CHUNK_COUNT:
    raise ValueError(f"預期 {EXPECTED_CHUNK_COUNT} 個 chunks，實際取得 {len(chunks)} 個。")

seen_ids = set()
for position, chunk in enumerate(chunks):
    if not isinstance(chunk, dict):
        raise TypeError(f"位置 {position} 的 chunk 必須是 JSON object。")

    missing = REQUIRED_FIELDS - chunk.keys()
    if missing: # 等價於 `if len(missing) > 0`，容器有沒有東西
        raise ValueError(f"位置 {position} 缺少欄位：{sorted(missing)}")

    for field in REQUIRED_FIELDS:
        if not isinstance(chunk[field], str) or not chunk[field].strip():
            raise ValueError(f"{chunk.get('id', position)} 的 {field} 必須是非空字串。")

    if chunk["id"] in seen_ids:
        raise ValueError(f"發現重複 ID：{chunk['id']}")
    seen_ids.add(chunk["id"])

    if chunk["content_type"] not in ALLOWED_CONTENT_TYPES:
        raise ValueError(
            f"{chunk['id']} 的 content_type 無效：{chunk['content_type']}"
        )

print(f"驗證通過：{len(chunks)} 個 chunks，ID 全部唯一。")
print(f"{'ID':<10} {'Title':<32} {'Type':<12} {'Chars':>7}")
print("-" * 66)
for chunk in chunks:
    print(
        f"{chunk['id']:<10} {chunk['title'][:31]:<32} "
        f"{chunk['content_type']:<12} {len(chunk['text']):>7}"
    )

驗證通過：10 個 chunks，ID 全部唯一。
ID         Title                            Type           Chars
------------------------------------------------------------------
L37-001    Overview                         canonical        475
L37-002    Architecture                     canonical       1089
L37-003    Water Properties                 canonical        940
L37-004    Acoustic Anomalies               canonical        595
L37-005    Entities                         canonical        315
L37-006    Settlements                      canonical        365
L37-007    Recovered Journal                narrative       1352
L37-008    Recovered Journal (continued)    narrative       1442
L37-009    Entrances                        canonical        324
L37-010    Exits and Hazards                canonical        571


## 4. 初始化多語言嵌入模型

`paraphrase-multilingual-MiniLM-L12-v2` 支援英文與繁體中文查詢。

Sentence Transformers 會依 Colab 環境自動使用 CUDA 或 CPU。

? 為什麼是CUDA或CPU

目前 notebook 主要針對 Google Colab，而 Colab 的 GPU 通常是 NVIDIA，因此只需檢查 cuda 和 cpu。
實際上有
```
"cuda"  # NVIDIA GPU
"mps"   # Apple Silicon GPU
"cpu"   # CPU
```

MPS 是 macOS 本機環境使用的，Google Colab 不會用到。對這份以 Colab 為主的作業而言，原本的寫法較簡潔且足夠。

In [4]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
NORMALIZE_EMBEDDINGS = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"Embedding model：{MODEL_NAME}")
print(f"執行裝置：{DEVICE}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model：sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
執行裝置：cpu


## 5. 產生文件embedding

每筆文件使用 `title + "\n" + text` 進行embedding。向量會正規化為單位長度並轉成 FAISS 所需的 `float32`。

batch_size（批次大小）表示模型一次同時處理多少筆資料。
`batch_size = 32`意思是模型每次最多處理 32 個文件，再繼續處理下一批。

`convert_to_numpy=True` 表示把模型輸出的 embeddings 轉成 NumPy 陣列。

使用 NumPy 的原因是 FAISS 能方便地接收 NumPy 陣列，而且通常要求向量為 float32

In [5]:
import numpy as np

documents = [f"{chunk['title']}\n{chunk['text']}" for chunk in chunks]
embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
).astype(np.float32)

if embeddings.ndim != 2 or embeddings.shape[0] != len(chunks):
    raise RuntimeError("嵌入矩陣形狀與 chunk 數量不一致。")
    # 正常的 embedding 矩陣應是二維：
    # (文件數量, 每個向量的維度)
if NORMALIZE_EMBEDDINGS and not np.allclose(
    np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5
):
    raise RuntimeError("部分文件向量未正規化為單位長度。")

embedding_dimension = int(embeddings.shape[1])
print(f"Embedding matrix shape：{embeddings.shape}")
print(f"dtype：{embeddings.dtype}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape：(10, 384)
dtype：float32


## 6. 建立 FAISS 索引

正規化向量的內積等同餘弦相似度，因此使用精確搜尋的 `faiss.IndexFlatIP`。向量加入順序與 `chunks` list 完全相同。

---

餘弦相似度的公式是：$\cos(A,B) = \frac{AB}{|A||B|} = AB$

正規化後，兩個向量的長度都是 1

因此，只要文件向量和查詢向量都先正規化，直接計算內積，就能得到餘弦相似度

In [6]:
import faiss

index = faiss.IndexFlatIP(embedding_dimension)
index.add(np.ascontiguousarray(embeddings, dtype=np.float32))

if index.ntotal != len(chunks):
    raise RuntimeError("FAISS 向量數量與 chunk 數量不一致。")

index_position_to_chunk_id = {position: chunk["id"] for position, chunk in enumerate(chunks)}
print(f"FAISS index：{type(index).__name__}")
print(f"向量數量：{index.ntotal}")
print("位置對應：", index_position_to_chunk_id)

FAISS index：IndexFlatIP
向量數量：10
位置對應： {0: 'L37-001', 1: 'L37-002', 2: 'L37-003', 3: 'L37-004', 4: 'L37-005', 5: 'L37-006', 6: 'L37-007', 7: 'L37-008', 8: 'L37-009', 9: 'L37-010'}


## 7. 定義檢索函式

`retrieve(query, top_k)` 使用同一模型與正規化設定編碼查詢，並依 FAISS 回傳的位置取回原始 chunk。空白查詢或不合法的 `top_k` 會得到清楚的錯誤訊息。

In [7]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """以餘弦相似度搜尋，回傳 chunk 與 similarity_score。"""
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query 必須是非空字串。")
    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k 必須是大於 0 的整數。")

    effective_top_k = min(top_k, index.ntotal)
    query_vector = model.encode(
        [query.strip()],
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        show_progress_bar=False,
    ).astype(np.float32)

    scores, positions = index.search(
        np.ascontiguousarray(query_vector), effective_top_k
    )

    results = []
    for score, position in zip(scores[0], positions[0]):
        if position < 0:
            continue
        result = dict(chunks[int(position)])
        result["similarity_score"] = float(score)
        result["index_position"] = int(position)
        results.append(result)
    return results

## 8. 英文與中文檢索實驗

以下測試直接顯示模型的實際 Top-3 結果，不指定或偽造預期命中。相似度越高代表查詢與文件向量越接近。

In [8]:
TEST_QUERIES = [
    "What is in the water of Level 37?",
    "Are there any entities in Level 37?",
    "How can I exit Level 37?",
    "Level 37 的水有什麼特別的地方？",
    "Level 37 有哪些出口？",
    "What are Scapes?",
]

def display_results(query: str, results: list[dict]) -> None:
    print(f"\nQuery: {query}")
    for rank, result in enumerate(results, start=1):
        preview = textwrap.shorten(
            " ".join(result["text"].split()), width=180, placeholder="…"
        )
        print(
            f"  {rank}. {result['id']} — {result['title']} "
            f"(score={result['similarity_score']:.4f})"
        )
        print(f"     {preview}")

for test_query in TEST_QUERIES:
    display_results(test_query, retrieve(test_query, top_k=3))


Query: What is in the water of Level 37?
  1. L37-001 — Overview (score=0.6200)
     SURVIVAL DIFFICULTY: 0 * Safe. * Unsecure. * Devoid of Entities. Level 37 is an expansive complex of interconnected rooms and corridors submerged in undulating, lukewarm water.…
  2. L37-009 — Entrances (score=0.5924)
     ## Entrances * Submerging oneself in the waters of Level 58 and Level 119 may lead to one reemerging in this level; * Porcelain-tiled areas of Level 233 infrequently lead to this…
  3. L37-010 — Exits and Hazards (score=0.5558)
     ## Exits * Immersing oneself in the water of Level 37 may cause reemergence in the pools of Level 130; * Enclosed, dark tunnels in this level may lead to Level 43; * Corridors in…

Query: Are there any entities in Level 37?
  1. L37-005 — Entities (score=0.6983)
     Level 37 is thought to be entirely devoid of life. No encounters with entities or other wanderers have been recorded in the level; it is unknown if this results from some…
  2. L37-001 — Ove

### 自訂查詢

執行下方 cell，輸入英文或繁體中文問題以觀察實際檢索結果。

In [18]:
user_query = input("請輸入查詢（English / 繁體中文）：").strip()
try:
    display_results(user_query, retrieve(user_query, top_k=3))
except ValueError as exc:
    print(f"輸入錯誤：{exc}")

請輸入查詢（English / 繁體中文）： Scapes 是什麼？

Query: Scapes 是什麼？
  1. L37-008 — Recovered Journal (continued) (score=0.2715)
     It is not the doing of any force or overseeing entity whatsoever; instead, it is one’s emotions that keep the rooms extending — the impetus for their continued growth and…
  2. L37-003 — Water Properties (score=0.1772)
     Investigation has shown that traces of Epsom salt (magnesium sulfate) are present in the waters of Level 37. This compound, when engulfing one’s body for extended periods of time,…
  3. L37-001 — Overview (score=0.1680)
     SURVIVAL DIFFICULTY: 0 * Safe. * Unsecure. * Devoid of Entities. Level 37 is an expansive complex of interconnected rooms and corridors submerged in undulating, lukewarm water.…


## 9. 匯出向量資料庫

輸出包含 FAISS binary index、與索引位置一一對應的原始 chunks，以及重建查詢流程所需的 embedding 設定。最後封裝成 `rag-data.zip` 並在 Colab 觸發下載。

In [19]:
OUTPUT_DIR = Path("rag_database")
ZIP_BASENAME = "rag-data"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

faiss.write_index(index, str(OUTPUT_DIR / "index.faiss"))
with (OUTPUT_DIR / "chunks.json").open("w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": embedding_dimension,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "similarity_metric": "cosine_similarity_via_inner_product",
    "chunk_count": len(chunks),
    "format_version": "1.0",
    "indexed_text_template": "{title}\n{text}",
}
with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, ensure_ascii=False, indent=2)

zip_path = Path(shutil.make_archive(ZIP_BASENAME, "zip", root_dir=".", base_dir=OUTPUT_DIR.name))
expected_files = {"index.faiss", "chunks.json", "config.json"}
actual_files = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
if actual_files != expected_files:
    raise RuntimeError(f"輸出檔案不完整：{sorted(actual_files)}")

print(f"已建立：{zip_path.resolve()}")
print("rag_database 內容：", sorted(actual_files))

try:
    from google.colab import files
except ImportError:
    print("目前不是 Colab 環境，請從上方路徑取得 ZIP。")
else:
    files.download(str(zip_path))

已建立：/content/rag-data.zip
rag_database 內容： ['chunks.json', 'config.json', 'index.faiss']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. 完成摘要與 Phase 3 使用方式

本筆記本完成資料驗證、多語言嵌入、FAISS 精確向量索引、雙語檢索測試與 `rag-data.zip` 匯出。

Phase 3 的 RAG02 應下載並解壓此 ZIP，讀取 `config.json` 以載入相同的 embedding model，使用 `faiss.read_index()` 載入 `index.faiss`，並以相同順序讀取 `chunks.json`。RAG02 只需嵌入使用者查詢，不必重新嵌入文件。